# Create NMR features

Attach every NMR-derived feature block to a raw spectra dataset and save a featurized pickle for the modeling notebooks.

**Input:** a dataset with `h_nmr_peaks` / `c_nmr_peaks` peak lists (registry name or path).  
**Output:** `Datasets/<name>_featurized.pkl` with An 2014 1H descriptors, 13C bins, peak summary stats, and (optionally) NMF dictionary codes.

All feature logic lives in [`nmrlib.features`](nmrlib/features.py) so the definitions never drift between notebooks.

In [9]:
import pandas as pd

from nmrlib import load_dataset, featurize
from nmrlib.features import feature_sets, nmf_cols

## Available datasets

Registry from [`nmrlib.data`](nmrlib/data.py). Descriptions are hardcoded labels; `found_in` is checked live against the filesystem. The column listing printed after loading (below) is the sanity check that a dataset's contents actually match its description.

In [10]:
from nmrlib import describe_datasets

describe_datasets()

,description,file,found_in
name,,,
alberts_10k,Alberts et al. 10k subset merged with qchem ta...,alberts_nmr_qchem_merged.pkl,Datasets/
alberts_10k_logp,"Alberts 10k with logP; raw spectra, pre-featur...",alberts_merged_10k_with_logp.pkl,Downloads/
alberts_10k_100kdict,Alberts 10k with NMF codes from the dictionary...,alberts_10k_100kdict_nmf_features.pkl,Downloads/
ids_nmr_1k,"IDS NMR corpus, 1k unlabeled spectra for dicti...",ids_nmr_1k.pkl,Downloads/
ids_nmr_10k,"IDS NMR corpus, 10k unlabeled spectra for dict...",ids_nmr_10k.pkl,Downloads/
ids_nmr_100k,"IDS NMR corpus, 100k unlabeled spectra for dic...",ids_nmr_100k.pkl,Downloads/
ids_1k_featurized,IDS 1k fully featurized (115-component NMF + A...,ids_1k_nmf_115_and_other.pkl,Downloads/
ids_1k_tuned_nmf,IDS 1k with tuned-NMF dictionary codes.,ids_nmr_1k_tuned_nmf_features.pkl,Downloads/
gaussian_1k,Gaussian-matched 1k set with NMR spectra.,gaussian_nmr_matched_1k.pkl,Datasets/


## Config

In [11]:
# Raw dataset (registry short name or path to a .pkl)
dataset = "nmrshiftdb"
out_path = "Datasets/nmrshiftdb_featurized.pkl"

# Optional pretrained NMF dictionary artifact to project spectra into.
# Set to None to skip NMF codes (An 2014 + 13C + stats only).
nmf_dictionary = "/Users/jacknugent/Downloads/gap_from_100k_dict/nmf_dictionary_100k.pkl"

## Load & inspect

In [12]:
df = load_dataset(dataset)
df.head()
print(f"Columns ({len(df.columns)}): " + ", ".join(map(str, df.columns)))

Loaded /Users/jacknugent/Downloads/nmrshift_13C_1H_feat 2.pkl — 3357 rows x 169 columns
Columns (169): INChI key, INChI_x, nmrshiftdb2 ID_x, rdkit_mol_x, smiles, molecular_formula_x, mol_weight_x, num_heavy_atoms_x, num_carbons_x, num_hydrogens_x, Temperature [K]_x, Field Strength [MHz]_x, Solvent_x, spectrum_13C, spectrum_13C_length, spectrum_13C_min, spectrum_13C_max, spectrum_13C_mean, spectrum_13C_median, spectrum_13C_mode, spectrum_13C_var, spectrum_13C_std, spectrum_13C_skewness, spectrum_13C_kurtosis, INChI_y, nmrshiftdb2 ID_y, rdkit_mol_y, smiles_y, molecular_formula_y, mol_weight_y, num_heavy_atoms_y, num_carbons_y, num_hydrogens_y, Temperature [K]_y, Field Strength [MHz]_y, Solvent_y, spectrum_1H, spectrum_1H_length, spectrum_1H_min, spectrum_1H_max, spectrum_1H_mean, spectrum_1H_median, spectrum_1H_mode, spectrum_1H_var, spectrum_1H_std, spectrum_1H_skewness, spectrum_1H_kurtosis, fr_Al_COO, fr_Al_OH, fr_Al_OH_noTert, fr_ArN, fr_Ar_COO, fr_Ar_N, fr_Ar_NH, fr_Ar_OH, fr_COO, f

In [13]:
# Peek at the raw peak lists that drive every feature block
print(df[["h_nmr_peaks", "c_nmr_peaks"]].iloc[0].to_dict())

{'h_nmr_peaks': [{'delta': 1.88, 'nH': 1.0}, {'delta': 1.88, 'nH': 1.0}, {'delta': 1.88, 'nH': 1.0}, {'delta': 5.78, 'nH': 1.0}, {'delta': 6.22, 'nH': 1.0}, {'delta': 6.22, 'nH': 1.0}, {'delta': 7.34, 'nH': 1.0}], 'c_nmr_peaks': [{'delta (ppm)': 18.8, 'intensity': 1.0}, {'delta (ppm)': 117.9, 'intensity': 1.0}, {'delta (ppm)': 129.7, 'intensity': 1.0}, {'delta (ppm)': 140.8, 'intensity': 1.0}, {'delta (ppm)': 147.4, 'intensity': 1.0}, {'delta (ppm)': 171.9, 'intensity': 1.0}]}


## Featurize

In [14]:
featurized = featurize(df, dictionary_path=nmf_dictionary)
print(f"{featurized.shape[0]} rows x {featurized.shape[1]} columns")
print(f"NMF code columns: {len(nmf_cols(featurized))}")

3357 rows x 346 columns
NMF code columns: 93


In [15]:
# Confirm the named feature sets the modeling notebooks expect are all present
for name, cols in feature_sets(featurized).items():
    print(f"{name:<28} {len(cols)} cols")

an14                         13 cols
an14_full                    28 cols
nmr_stats                    8 cols
NMF                          93 cols
c_bins_10                    26 cols
c_bins_5                     48 cols
an14_full_with_c_bins_10     54 cols
an14_full_with_c_bins_5      76 cols
NMF_with_nmr_stats           101 cols
NMF_with_c_bins_5            141 cols
an14_full_with_nmr_stats     36 cols


## Save

In [16]:
featurized.to_pickle(out_path)
print(f"Saved -> {out_path}")

Saved -> Datasets/nmrshiftdb_featurized.pkl
